# IEAP – Python series 03: Assignment - Group Lagarde | Long | Deemu

| Part | Student | Sections | Branch |
|---|---|---|---|
|  **A** | **LAGARDE Alexis** (Student A) | Section 2 - find and plot remarkable points (2.1, 2.2, 2.3) + common setup | `part-a` |
|  **B** | **LONG Jiangbin** (Student B) | Section 3 - analyze a known signal (3.1, 3.2, 3.3) | `part-b` |
|  **C** | **DEEMU Charles Sam** (Student C) | Section 4 - noisy signal and low-pass filter (4.1, 4.2, 4.3) | `part-c` |

Working on different files were thought, but as for part B we need function from part A and same for part C, we decided to work on the same file with for every part a new branch based on a up to date `main` branch

# PART A - LAGARDE Alexis
**Section 2 – functions to find and plot remarkable points (2.1, 2.2, 2.3)**

### Imports
Legends aren't displaying correctly if `matplotlib widget` option is uncommented, clear all outputs and restart the kernel if plots are still zoomable (used for interactive plots in the notebook)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
# %matplotlib widget   

## 2. Create functions to find and plot remarkable points

Reference signal for the tests:

In [ ]:
index = np.arange(10)
signal = np.array([-1, 1, 2, 1, 0, -1, 1, 2, 1, -2])
print("  index = ", index.tolist())
print(" signal = ", signal.tolist())

### 2.1. Functions to find zero crossings and to plot them

The functions below are those of the live-coding session, **with the docstrings and comments improved as asked in 2.2**.

**Definition used:** a crossing is reported at index `i` when the sign changes between the last non-zero sample
at (or before) `i` and sample `i + 1`. Samples equal to 0 have no sign of their own: they inherit the sign of the
previous non-zero sample (leading zeros take the sign of the first non-zero sample).
So `[-1, 0, 0, 1]` has one positive crossing at index 2, `[1, 0, 1]` has none, and an all-zero signal has none.

In [ ]:
def find_zero_crossings_no_zeros(signs: np.ndarray):
    """Find positive and negative zero crossings in a sign array WITHOUT zeros.

    Parameters
    ----------
    signs : np.ndarray of int, shape (n,)
        Sign of each sample, containing only -1 and +1 (no 0). Use
        :func:`propagate_signs_over_zeros` first if the array may contain 0.

    Returns
    -------
    i_cross_pos : np.ndarray of int
        Indices ``i`` such that ``signs[i] == -1`` and ``signs[i + 1] == +1``
        (the signal goes from negative to positive between ``i`` and ``i + 1``).
    i_cross_neg : np.ndarray of int
        Indices ``i`` such that ``signs[i] == +1`` and ``signs[i + 1] == -1``.
    """
    # diff[i] = signs[i+1] - signs[i] is +2 on a -1 -> +1 change,
    # -2 on a +1 -> -1 change, and 0 when the sign stays the same.
    diff = np.diff(signs)
    # np.where(...) returns a tuple (one array per dimension): take [0] for 1D
    i_cross_pos = np.where(diff > 0)[0]
    i_cross_neg = np.where(diff < 0)[0]
    return i_cross_pos, i_cross_neg

def propagate_signs_over_zeros(signs: np.ndarray):
    """Replace every 0 of a sign array by the nearest meaningful non-zero sign.

    * a 0 takes the sign of the closest non-zero sample **before** it
      (forward fill);
    * leading zeros (no non-zero sample before them) take the sign of the
      first non-zero sample (backward fill);
    * an array made only of zeros is returned unchanged.

    Parameters
    ----------
    signs : np.ndarray of int, shape (n,)
        Array of -1, 0, +1 (typically ``np.sign(s)``). It is not modified.

    Returns
    -------
    signs_no_zeros : np.ndarray of int, shape (n,)
        Copy of ``signs`` where zeros have been replaced as described above.

    Example
    -------
    >>> propagate_signs_over_zeros(np.array([0, 0, 1, 0, -1, 0]))
    array([ 1,  1,  1,  1, -1, -1])
    """
    signs_no_zeros = signs.copy()  # work on a copy: never modify the input

    # Forward fill: each zero gets the sign of the previous non-zero
    mask = signs_no_zeros != 0  # True where the sample has a sign of its own
    # For each position, keep its own index if it is non-zero, else 0.
    # e.g. signs = [ 1, 0, 0,-1, 0] -> idx = [0, 0, 0, 3, 0]
    idx = np.where(mask, np.arange(len(signs_no_zeros)), 0)
    # Running maximum: each position now holds the index of the LAST non-zero
    # sample at or before it.  idx = [0, 0, 0, 3, 3]
    idx = np.maximum.accumulate(idx)
    # Fancy indexing: copy the sign found at that index. -> [1, 1, 1, -1, -1]
    signs_no_zeros = signs_no_zeros[idx]

    # Backward fill of leading zeros
    # If the array starts with zeros, step 1 pointed them to index 0, which is
    # itself 0, so they are still 0. Give them the sign of the first non-zero.
    if signs_no_zeros[0] == 0:
        i_first_nonzero = np.where(signs != 0)[0]
        if len(i_first_nonzero) > 0:  # otherwise the array is all zeros
            signs_no_zeros[: i_first_nonzero[0]] = signs[i_first_nonzero[0]]

    return signs_no_zeros

def find_zero_crossings(s: np.ndarray):
    """Find the positive and negative zero crossings of a 1D signal.

    Parameters
    ----------
    s : array_like, shape (n,)
        The signal (list or numpy array, int or float).

    Returns
    -------
    i_cross_pos : np.ndarray of int
        Indices ``i`` where the signal goes from negative to positive between
        ``i`` and ``i + 1`` (rising edge, positive slope).
    i_cross_neg : np.ndarray of int
        Indices ``i`` where the signal goes from positive to negative between
        ``i`` and ``i + 1`` (falling edge, negative slope).

    Raises
    ------
    ValueError
        If ``s`` is not one-dimensional.

    Notes
    -----
    Samples equal to 0 are handled by :func:`propagate_signs_over_zeros`
    (see the module docstring for the exact definition).
    """
    # Input validation
    s = np.asarray(s)  # accept lists / tuples as well as arrays
    if s.ndim != 1:
        raise ValueError("Input must be a 1D array")
    if s.size < 2:  # a crossing needs at least two samples
        return np.array([], dtype=int), np.array([], dtype=int)

    # Keep only the sign of each sample: -1, 0 or +1 
    signs = np.sign(s).astype(int)

    # Remove zeros only if there are some (saves work otherwise) 
    if np.any(signs == 0):
        signs_no_zeros = propagate_signs_over_zeros(signs)
    else:
        signs_no_zeros = signs

    # A crossing is now simply a change of sign between neighbours 
    return find_zero_crossings_no_zeros(signs_no_zeros)

Plotting functions (`plot_` = no `plt.show()`, `display_` = shows):

In [ ]:
def plot_signal(i: np.ndarray, s: np.ndarray, color: str = "blue",
                label: str = "signal", xlabel: str = "index",
                ylabel: str = "signal"):
    """Plot a signal as dots joined by a thin line, with a horizontal zero line.

    Parameters
    ----------
    i : np.ndarray, shape (n,)
        x values (sample indices or time vector).
    s : np.ndarray, shape (n,)
        Signal values.
    color : str
        Matplotlib color of the signal.
    label : str
        Legend label.
    xlabel, ylabel : str
        Axis labels.

    Returns
    -------
    None. Draws on the current axes, does not show the figure.
    """
    plt.axhline(y=0, color="gray", linestyle="-")
    plt.plot(i, s, ".-", markersize=5, linewidth=0.25, color=color, label=label)
    plt.ylabel(ylabel)
    plt.xlabel(xlabel)

def plot_remarkable_points(t: np.ndarray, s: np.ndarray, format: str, label: str):
    """Plot a set of remarkable points with big markers.

    Parameters
    ----------
    t : np.ndarray
        x values of the points only (e.g. ``time[i_points]``).
    s : np.ndarray
        y values of the points only (e.g. ``signal[i_points]``).
    format : str
        Matplotlib format string, e.g. ``"^r"`` (red up-triangles).
    label : str
        Legend label.

    Returns
    -------
    None. Draws on the current axes, does not show the figure.
    """
    plt.plot(t, s, format, markersize=10, label=label)

def plot_positive_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot the positive zero crossings of ``s`` (red up-triangles).

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    i_pos, _ = find_zero_crossings(s)
    plot_remarkable_points(i[i_pos], s[i_pos], "^r", "positive zero crossings")

def plot_negative_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot the negative zero crossings of ``s`` (yellow down-triangles).

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    _, i_neg = find_zero_crossings(s)
    plot_remarkable_points(i[i_neg], s[i_neg], "vy", "negative zero crossings")

def plot_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot both positive and negative zero crossings of ``s``."""
    plot_positive_zero_crossings(i, s)
    plot_negative_zero_crossings(i, s)

def finalize_plot():
    """Add the legend (outside, on the right) and the grid, then show.

    NOTE: a legend outside the axes does not work well with
    ``%matplotlib widget``.
    """
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True)
    plt.show()

def display_signal_and_crossings(i: np.ndarray, s: np.ndarray):
    """Show a new figure with the signal and its zero crossings.

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    plt.figure()
    plot_signal(i, s)
    plot_zero_crossings(i, s)
    finalize_plot()

In [ ]:
display_signal_and_crossings(index, signal)
i_cross_pos, i_cross_neg = find_zero_crossings(signal)
print("positive zero crossings:", i_cross_pos, "  negative zero crossings:", i_cross_neg)

### 2.2. Explain and improve the previous code

**How `find_zero_crossings` works**

1. **Validate the input**: convert to a numpy array, reject anything that is not 1D, return empty results if there are fewer than 2 samples.
2. **Keep only the sign** of each sample: `np.sign` gives -1, 0 or +1.
3. **Remove the zeros** (`propagate_signs_over_zeros`):
   * *Forward fill*: `np.where(mask, np.arange(n), 0)` keeps the index of each non-zero sample (0 elsewhere), then `np.maximum.accumulate` turns it into "index of the last non-zero sample so far"; indexing `signs[idx]` copies that sign onto the zeros - no Python loop needed;
   * *Backward fill*: leading zeros still point to index 0 (which is 0), so they get the sign of the first non-zero sample.
4. **Detect sign changes** (`find_zero_crossings_no_zeros`): `np.diff(signs)` equals +2 on a − → + change (positive crossing), −2 on a + → − change (negative crossing), 0 otherwise.

**Improvements made**
* NumPy-style docstrings for every function: what it does, parameters, returns, raised errors, example.
* A comment on every line that is not straightforward, and a comment per block (validation / signs / zeros / crossings).
* Early return for signals shorter than 2 samples; `np.any(signs == 0)` instead of building an unused index array.
* `display_signal_and_crossings` now reuses `plot_zero_crossings` and a common `finalize_plot` (legend + grid + show);
  `plot_signal` has optional `label`, `xlabel`, `ylabel` so it can be reused with a time axis in parts B and C.

**Role of the unit tests.** A unit test runs a function on a tiny input whose answer is known by hand and checks it with `assert`.
The tests:
* **document the definition** of a zero crossing with concrete examples (zeros inside a crossing, a simple touch of zero, zeros at the borders);
* **catch regressions**: if someone changes the code later and breaks a case, a test fails immediately instead of silently giving wrong plots;
* **cover edge cases** rarely met in real signals but that can crash or mislead the code.

**Tests added** to cover all scenarios: the reference signal; always positive / always negative; touching zero from above and from below;
all zeros; empty, 1-sample and 2-sample signals; crossings in the first and last pair; float signals with tiny values and tuple input;
a sampled sine (period = 100 samples); 2D input must raise `ValueError`; the input must not be modified; direct tests of both helper functions.

In [ ]:

# helper used by the tests
def check(s, expected_pos, expected_neg):
    """Helper: compare the output of find_zero_crossings with the expectation."""
    i_pos, i_neg = find_zero_crossings(s)
    assert np.array_equal(i_pos, np.array(expected_pos, dtype=int)), (s, i_pos)
    assert np.array_equal(i_neg, np.array(expected_neg, dtype=int)), (s, i_neg)


# The 6 cases given in the assignment
def test_given_cases():
    check([-1, 1, -1, 1, -1], [0, 2], [1, 3])     # alternating, no zeros
    check([-1, 0, 0, 1, 0, -1], [2], [4])         # zeros inside crossings
    check([0, 0, 0, 1, 0, 0], [], [])             # only touches zero
    check([0, 0, 1, 0, -1, 0], [], [3])           # leading and trailing zeros
    check([0, 1, 0, 0, -1, 0], [], [3])
    check([1, 0, 0, -1, 0, 0], [], [2])

In [ ]:
# Cases added by the group
def test_reference_signal():
    """The reference signal of the assignment (section 2)."""
    check([-1, 1, 2, 1, 0, -1, 1, 2, 1, -2], [0, 5], [4, 8])

def test_no_crossing_constant_sign():
    check([1, 2, 3, 2, 1], [], [])                # always positive
    check([-1, -2, -3], [], [])                   # always negative

def test_touch_zero_without_crossing():
    check([1, 0, 1], [], [])                      # touches from above
    check([-1, 0, 0, -1], [], [])                 # touches from below

def test_all_zeros():
    check([0, 0, 0, 0], [], [])

def test_degenerate_lengths():
    check([], [], [])                             # empty signal
    check([5], [], [])                            # single sample
    check([0], [], [])
    check([-1, 1], [0], [])                       # shortest possible crossing

def test_crossing_at_the_borders():
    check([1, -1, -1, -1], [], [0])               # first pair
    check([-1, -1, -1, 1], [2], [])               # last pair

def test_float_signal_and_list_input():
    check([-0.5, 0.25, -1e-12, 3.0], [0, 2], [1])  # floats, tiny values
    check((-1, 1), [0], [])                        # tuple input accepted

def test_sine_wave():
    """Sampled 1 Hz sine over 2 s: two crossings of each kind, 1 period apart."""
    t = np.arange(0, 2, 0.01)
    s = np.sin(2 * np.pi * t + 0.1)                # phase avoids exact zeros
    i_pos, i_neg = find_zero_crossings(s)
    assert len(i_pos) == 2 and len(i_neg) == 2
    assert np.array_equal(np.diff(i_pos), [100])
    assert np.all(s[i_pos] < 0) and np.all(s[i_pos + 1] >= 0)

def test_rejects_2d_input():
    try:
        find_zero_crossings(np.zeros((2, 3)))
    except ValueError:
        return
    raise AssertionError("ValueError not raised for a 2D input")

def test_input_not_modified():
    s = np.array([0, 1, 0, -1])
    signs = np.sign(s)
    propagate_signs_over_zeros(signs)
    assert np.array_equal(signs, [0, 1, 0, -1])

def test_propagate_signs_over_zeros():
    assert np.array_equal(propagate_signs_over_zeros(np.array([0, 0, 1, 0, -1, 0])),
                          [1, 1, 1, 1, -1, -1])
    assert np.array_equal(propagate_signs_over_zeros(np.array([-1, 0, 0, 1])),
                          [-1, -1, -1, 1])
    assert np.array_equal(propagate_signs_over_zeros(np.array([0, 0])), [0, 0])

def test_find_zero_crossings_no_zeros():
    i_pos, i_neg = find_zero_crossings_no_zeros(np.array([1, -1, -1, 1]))
    assert np.array_equal(i_pos, [2]) and np.array_equal(i_neg, [0])

In [ ]:
def run_tests(tests):
    """Run a list of test functions; stop with the AssertionError if one fails."""
    for test in tests:
        test()
    print(f"All {len(tests)} tests passed.")


run_tests([test_given_cases, test_reference_signal, test_no_crossing_constant_sign,
           test_touch_zero_without_crossing, test_all_zeros, test_degenerate_lengths,
           test_crossing_at_the_borders, test_float_signal_and_list_input, test_sine_wave,
           test_rejects_2d_input, test_input_not_modified, test_propagate_signs_over_zeros,
           test_find_zero_crossings_no_zeros])

### 2.3. Create new functions to find the local maxima and minima

A local extremum is a zero crossing of the first derivative. `d = np.diff(s)` gives `d[k] = s[k+1] - s[k]`, the slope between `k` and `k+1`:
* slope + → − (negative crossing of `d`) ⇒ **local maximum** at `k + 1`;
* slope − → + (positive crossing of `d`) ⇒ **local minimum** at `k + 1`.

Since `find_zero_crossings` propagates signs over zeros, plateaus are handled too (one extremum, on the last sample of the plateau).

In [ ]:
def find_local_extrema(s: np.ndarray):
    """Find the local maxima and minima of a 1D signal.

    Parameters
    ----------
    s : array_like, shape (n,)
        The signal.

    Returns
    -------
    i_local_max : np.ndarray of int
        Indices of the local maxima (``s[i_local_max]`` are the maximum values).
    i_local_min : np.ndarray of int
        Indices of the local minima.

    Notes
    -----
    The first and last samples are never reported (their slope on one side
    is unknown).
    """
    s = np.asarray(s)
    derivative = np.diff(s)  # approximate first derivative (one sample shorter)
    i_d_pos, i_d_neg = find_zero_crossings(derivative)
    # +1 : crossing between d[k] and d[k+1] <=> extremum on sample k+1 of s
    i_local_max = i_d_neg + 1
    i_local_min = i_d_pos + 1
    return i_local_max, i_local_min

def find_local_maxima(s: np.ndarray):
    """Return the indices of the local maxima of ``s``."""
    return find_local_extrema(s)[0]

def find_local_minima(s: np.ndarray):
    """Return the indices of the local minima of ``s``."""
    return find_local_extrema(s)[1]

def plot_local_maxima(i: np.ndarray, s: np.ndarray, format: str = "*r",
                      label: str = "local maxima"):
    """Find and plot the local maxima of ``s`` (red stars by default)."""
    i_max = find_local_maxima(s)
    plot_remarkable_points(i[i_max], s[i_max], format, label)

def plot_local_minima(i: np.ndarray, s: np.ndarray, format: str = "*m",
                      label: str = "local minima"):
    """Find and plot the local minima of ``s`` (magenta stars by default)."""
    i_min = find_local_minima(s)
    plot_remarkable_points(i[i_min], s[i_min], format, label)

def plot_local_extrema(i: np.ndarray, s: np.ndarray):
    """Find and plot both local maxima and minima of ``s``."""
    plot_local_maxima(i, s)
    plot_local_minima(i, s)

def display_signal_and_extrema(i: np.ndarray, s: np.ndarray):
    """Show a new figure with the signal and its local maxima / minima."""
    plt.figure()
    plot_signal(i, s)
    plot_local_extrema(i, s)
    finalize_plot()

In [ ]:
i_local_max, i_local_min = find_local_extrema(signal)
print("local maxima:", i_local_max, "  local minima:", i_local_min)
display_signal_and_extrema(index, signal)

In [ ]:
def test_reference_signal_extrema():
    i_max, i_min = find_local_extrema([-1, 1, 2, 1, 0, -1, 1, 2, 1, -2])
    assert np.array_equal(i_max, [2, 7])
    assert np.array_equal(i_min, [5])

def test_monotonic_has_no_extrema():
    i_max, i_min = find_local_extrema([1, 2, 3, 4])
    assert i_max.size == 0 and i_min.size == 0

def test_plateau_gives_one_extremum():
    i_max, i_min = find_local_extrema([0, 1, 1, 0, -1, -1, 0])
    assert np.array_equal(i_max, [2])   # last sample of the top plateau
    assert np.array_equal(i_min, [5])


run_tests([test_reference_signal_extrema, test_monotonic_has_no_extrema, test_plateau_gives_one_extremum])

# PART B - LONG Jiangbin (Student B)
**Section 3 - analyze a known signal (3.1, 3.2, 3.3)**

## 3. Analyze a known signal
### 3.1. Create the signal and plot it

The signal `s` is the sum of two sine waves:

$$ s(t) = s_1(t) + s_2(t) = \sin(2\pi \cdot 2t) + 2\sin(2\pi \cdot t + 1) $$

| | Frequency | Amplitude | Phase |
|---|---|---|---|
| $s_1$ | 2 Hz | 1 | 0 |
| $s_2$ | 1 Hz | 2 | 1 rad |

It is sampled at 100 Hz from 0 to 2 s.

The formula is not given in the assignment: the frequencies and amplitudes were read on the figure,
and the phase was chosen so that our results match the expected ones.

In [ ]:
def create_time_vector(duration: float = 2.0, fs: float = 100.0):
    """Return a time vector from 0 to ``duration`` (both included) at ``fs`` Hz.

    Example: duration=2, fs=100 -> 201 samples 0.00, 0.01, ..., 2.00.
    """
    n_samples = int(round(duration * fs)) + 1
    return np.arange(n_samples) / fs  # exact multiples of 1/fs, no float drift

def create_sine(t: np.ndarray, frequency: float, amplitude: float = 1.0,
                phase: float = 0.0):
    """Return ``amplitude * sin(2*pi*frequency*t + phase)`` (phase in radians)."""
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)

def create_known_signal(t: np.ndarray):
    """Return the homework signal and its two components.

    s1 = sin(2*pi*2*t)            (2 Hz, amplitude 1)
    s2 = 2 * sin(2*pi*1*t + 1)    (1 Hz, amplitude 2, phase 1 rad)
    s  = s1 + s2

    Returns
    -------
    s, s1, s2 : np.ndarray, same shape as ``t``
    """
    s1 = create_sine(t, frequency=2.0, amplitude=1.0)
    s2 = create_sine(t, frequency=1.0, amplitude=2.0, phase=1.0)
    return s1 + s2, s1, s2

def display_signal_components(t, s, s1, s2):
    """Show s1 (green), s2 (orange) and s (blue) on the same figure."""
    plt.figure()
    plt.plot(t, s1, ".-", color="green", markersize=3, linewidth=0.25, label="s1")
    plt.plot(t, s2, ".-", color="orange", markersize=3, linewidth=0.25, label="s2")
    plt.plot(t, s, ".-", color="blue", markersize=6, linewidth=0.25, label="s")
    plt.xlabel("time [s]")
    plt.ylabel("signal")
    finalize_plot()


In [ ]:
fs = 100.0                                     # sampling frequency (Hz)
t = create_time_vector(duration=2.0, fs=fs)    # 0.00, 0.01, ..., 2.00
s, s1, s2 = create_known_signal(t)
display_signal_components(t, s, s1, s2)

### 3.2. Plot the remarkable points

We reuse the functions of Part A (extrema and zero crossings).

In [ ]:
def plot_all_remarkable_points(t, s):
    """Plot local maxima, local minima and zero crossings of ``s``."""
    plot_local_extrema(t, s)
    plot_zero_crossings(t, s)

def display_signal_and_remarkable_points(t, s, xlabel="time [s]"):
    """Show a new figure with the signal and all its remarkable points."""
    plt.figure()
    plot_signal(t, s, xlabel=xlabel)
    plot_all_remarkable_points(t, s)
    finalize_plot()

In [ ]:
display_signal_and_remarkable_points(t, s)

### 3.3. Use the remarkable points to find the frequency of the signal

A periodic signal crosses zero in the same direction once per period, so the time between
two **positive** zero crossings (or two **negative** ones) is the period.

1. Spacing between same-slope crossings, in samples (`np.diff`)
2. Period = spacing × sampling period (0.01 s)
3. Final period = mean of the positive and negative estimates
4. Frequency = 1 / period

In [ ]:
def compute_frequency_from_zero_crossings(t: np.ndarray, s: np.ndarray):
    """Estimate the period and frequency of ``s`` from same-slope zero crossings.

    Two successive crossings with the same slope (two positive, or two
    negative) are one period apart.

    Parameters
    ----------
    t : np.ndarray, shape (n,)   time vector (uniform sampling), in s.
    s : np.ndarray, shape (n,)   signal.

    Returns
    -------
    dict with keys (NaN when fewer than 2 crossings of a kind are found):
        "delta_zero_crossing_pos (samples)" : spacing of positive crossings
        "delta_zero_crossing_neg (samples)" : spacing of negative crossings
        "signal_period_pos (s)"  : mean period from positive crossings
        "signal_period_neg (s)"  : mean period from negative crossings
        "signal_period (s)"      : mean of the two periods above
        "signal_frequency (Hz)"  : 1 / signal_period
    """
    dt = t[1] - t[0]  # sampling period
    i_pos, i_neg = find_zero_crossings(s)
    delta_pos = np.diff(i_pos)  # distance (in samples) between same-slope crossings
    delta_neg = np.diff(i_neg)
    period_pos = np.mean(delta_pos) * dt if delta_pos.size else np.nan
    period_neg = np.mean(delta_neg) * dt if delta_neg.size else np.nan
    period = np.nanmean([period_pos, period_neg])
    return {
        "delta_zero_crossing_pos (samples)": delta_pos,
        "delta_zero_crossing_neg (samples)": delta_neg,
        "signal_period_pos (s)": float(period_pos),
        "signal_period_neg (s)": float(period_neg),
        "signal_period (s)": float(period),
        "signal_frequency (Hz)": float(1 / period),
    }

def print_frequency_info(info: dict, title=None):
    """Pretty-print the dictionary returned by
    :func:`compute_frequency_from_zero_crossings`, keys right-aligned."""
    if title:
        print(title)
    for key, value in info.items():
        print(f"  {key:>33} : {value}")

In [ ]:
info_clean = compute_frequency_from_zero_crossings(t, s)
print_frequency_info(info_clean)

In [ ]:
def test_known_signal():
    t = create_time_vector(2.0, 100.0)
    s, _, _ = create_known_signal(t)
    assert len(t) == 201
    assert np.isclose(np.ptp(s), 4.822252617910321)
    i_max, i_min = find_local_extrema(s)
    assert len(i_max) == 4 and len(i_min) == 4

def test_frequency_known_signal():
    t = create_time_vector(2.0, 100.0)
    s, _, _ = create_known_signal(t)
    info = compute_frequency_from_zero_crossings(t, s)
    assert np.array_equal(info["delta_zero_crossing_pos (samples)"], [100])
    assert np.array_equal(info["delta_zero_crossing_neg (samples)"], [100])
    assert np.isclose(info["signal_frequency (Hz)"], 1.0)

def test_frequency_pure_sine():
    t = create_time_vector(3.0, 1000.0)
    s = np.sin(2 * np.pi * 5 * t + 0.3)
    info = compute_frequency_from_zero_crossings(t, s)
    assert np.isclose(info["signal_frequency (Hz)"], 5.0, rtol=1e-2)


run_tests([test_known_signal, test_frequency_known_signal, test_frequency_pure_sine])

# PART C - DEEMUA Charles Sam (Student C)
**Section 4 - redo the analysis with a noisy signal (4.1, 4.2, 4.3)**


## 4. Redo the analysis with a noisy signal
### 4.1. Create the noisy  signal and plot it

We add **white noise** to the signal: each noise value is random and independent of the others,
and all frequencies are present with the same strength.

- Generated with `np.random` (uniform distribution)
- Noise amplitude = 20 % of the signal's peak-to-peak amplitude

The random seed is fixed so the results are the same every time we run the notebook.
Since the handout used a different random draw, our noisy results differ slightly from it.

In [ ]:
def create_white_noise(n_samples: int, noise_amplitude: float, seed=None):
    """Return uniform white noise with a peak-to-peak span of ``noise_amplitude``.

    Values are drawn with ``np.random.uniform`` in
    ``[-noise_amplitude/2, +noise_amplitude/2]``.

    Parameters
    ----------
    n_samples : int
    noise_amplitude : float   total span (max - min) of the noise.
    seed : int or None        fix it to get reproducible results.
    """
    rng = np.random.default_rng(seed)
    return rng.uniform(-noise_amplitude / 2, noise_amplitude / 2, n_samples)

def add_white_noise(s: np.ndarray, relative_amplitude: float = 0.2, seed=None):
    """Add white noise whose span is ``relative_amplitude`` x peak-to-peak of ``s``.

    Returns
    -------
    s_noisy : np.ndarray       s + noise
    noise : np.ndarray         the noise alone
    info : dict                peak-to-peak amplitude of s and noise amplitude
    """
    signal_peak_to_peak_amplitude = np.ptp(s)
    noise_amplitude = relative_amplitude * signal_peak_to_peak_amplitude
    noise = create_white_noise(len(s), noise_amplitude, seed)
    info = {"signal_peak_to_peak_amplitude": signal_peak_to_peak_amplitude,
            "noise_amplitude": noise_amplitude}
    return s + noise, noise, info

def display_noise(t, noise):
    """Show the noise alone as dots."""
    plt.figure()
    plt.plot(t, noise, ".")
    plt.xlabel("Time (s)")
    plt.ylabel("Noise amplitude")
    plt.show()

def display_signal_and_noisy_signal(t, s, s_noisy):
    """Show the clean signal (light gray) and the noisy one (orange dots +
    thin line, linewidth=0.25)."""
    plt.figure()
    plt.axhline(0, color="gray")
    plt.plot(t, s, ".-", color="lightgray", linewidth=0.25, label="signal")
    plt.plot(t, s_noisy, ".-", color="orange", linewidth=0.25, label="signal + noise")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True)
    plt.show()





In [ ]:
SEED = 7
s_noisy, noise, info_noise = add_white_noise(s, relative_amplitude=0.2, seed=SEED)
for key, value in info_noise.items():
    print(f"{key} = ", value)
display_noise(t, noise)
display_signal_and_noisy_signal(t, s, s_noisy)

### 4.2. Plot the remarkable points in the noisy signal

By hand, we expect roughly the points of the clean signal: two large maxima near 0.1 s and 1.1 s,
negative crossings near 0.3 s and 1.3 s, positive crossings near 0.9 s and 1.9 s.

With our functions, the noise changes the sign of the derivative almost at every sample, so nearly every sample becomes
a maximum or a minimum, and extra zero crossings appear wherever the signal hesitates around 0.

In [ ]:
def display_remarkable_points_of_noisy_signal(t, s, s_noisy, label="signal + noise"):
    """Show the clean signal, the noisy (or filtered) one and all the
    remarkable points found on the latter."""
    plt.figure()
    plot_signal(t, s, color="blue", label="signal", xlabel="Time (s)", ylabel="Amplitude")
    plt.plot(t, s_noisy, ".-", color="tab:blue", linewidth=1, markersize=3, label=label)
    plot_all_remarkable_points(t, s_noisy)
    finalize_plot()